# Window functions: Hann vs Tukey

When computing an FFT, the data is treated as if it tiles infinitely. Any hard edge creates a discontinuity that shows up as a bright cross-shaped artifact in the FFT.

This matters most in two places:
- **ROI FFT** - cropping a region creates new hard edges at the crop boundary. The Hann window removes them.
- **Image alignment** - correlating two images via FFT. The Tukey window removes edge artifacts while preserving 80% of the signal for an accurate correlation peak.

In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

In [ ]:
import numpy as np
import torch
from quantem.widget import Show2D
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Generate a crystal lattice

A synthetic crystal with two lattice spacings (12 px and 18 px). The periodic structure produces sharp Bragg spots in the FFT.

In [ ]:
N = 256
r, c = torch.meshgrid(
    torch.arange(N, device=device, dtype=torch.float32),
    torch.arange(N, device=device, dtype=torch.float32),
    indexing="ij",
)
lattice = (
    torch.sin(2 * torch.pi * c / 12) * torch.sin(2 * torch.pi * r / 12)
    + 0.5 * torch.sin(2 * torch.pi * (c + r) / 18)
    + 0.3 * torch.randn(N, N, device=device)
)
lattice = lattice.cpu().numpy()

## ROI FFT - where Hann windowing matters

Enable the ROI and FFT toggle below. When you draw an ROI and view its FFT, the cropped region has hard edges at the crop boundary. The Hann window (`fft_window=True`, the default) tapers these edges to zero, removing the cross artifact from the ROI FFT.

Try toggling the FFT Window switch on and off to see the difference.

In [ ]:
w = Show2D(lattice, title="Draw an ROI, then toggle FFT", show_fft=True, roi_active=True)
w.add_roi(row=80, col=80, shape="square")
w.roi_square(half_size=50)
w

## The Hann window

Smooth dome - maximum at center, zero at all edges. The 2D window is the outer product of two 1D windows: $W(r, c) = w(r) \cdot w(c)$.

$$w(n) = 0.5\left(1 - \cos\frac{2\pi n}{N-1}\right)$$

Used for ROI FFT display. The taper removes the hard crop boundary, giving a clean FFT of the selected region.

In [ ]:
hann_1d = np.hanning(N)
hann_2d = np.outer(hann_1d, hann_1d)
Show2D(hann_2d, title="Hann window (2D)", cmap="viridis")

## The Tukey window ($\alpha = 0.2$)

Flat plateau in the center (80% of the image at full strength), cosine tapers only on the outer 10% per side. Used for image alignment (Align2D, Align2DBulk) because it preserves the data signal for correlation while still removing edge artifacts.

$$w(t) = \begin{cases}
0.5\left(1 + \cos\frac{2\pi}{\alpha}\left(t - \frac{\alpha}{2}\right)\right) & \text{if } t < \frac{\alpha}{2} \\
1 & \text{if } \frac{\alpha}{2} \leq t \leq 1 - \frac{\alpha}{2} \\
0.5\left(1 + \cos\frac{2\pi}{\alpha}\left(t - 1 + \frac{\alpha}{2}\right)\right) & \text{if } t > 1 - \frac{\alpha}{2}
\end{cases}$$

where $t = n/(N-1) \in [0, 1]$.

In [ ]:
def tukey_1d(n, alpha=0.2):
    """1D Tukey window (same implementation as align2d.py)."""
    if n <= 1:
        return np.ones(n)
    t = np.linspace(0, 1, n)
    win = np.ones(n)
    left = t < alpha / 2
    right = t > 1 - alpha / 2
    win[left] = 0.5 * (1 + np.cos(2 * np.pi / alpha * (t[left] - alpha / 2)))
    win[right] = 0.5 * (1 + np.cos(2 * np.pi / alpha * (t[right] - 1 + alpha / 2)))
    return win
tukey_2d = np.outer(tukey_1d(N), tukey_1d(N))
Show2D(tukey_2d, title="Tukey window (α=0.2, 2D)", cmap="viridis")

## Effect on the image

Hann dims the entire image (dome shape), while Tukey keeps the center at full strength and only fades the edges.

In [ ]:
Show2D(lattice * hann_2d, title="Image x Hann window")

In [ ]:
Show2D(lattice * tukey_2d, title="Image x Tukey window (α=0.2)")

## Summary

| Window | Shape | Tapered region | Used for | Why |
|--------|-------|---------------|----------|-----|
| **Hann** | Smooth dome | Entire image | ROI FFT display | Removes hard crop boundary artifacts |
| **Tukey** ($\alpha=0.2$) | Flat plateau + edge taper | Outer 10% per side | Image alignment | Preserves 80% of signal for correlation |